# 1. RAFT training data generation using GPT-4o

In this notebook, we'll synthesize some training data that will eventually be used to fine-tune a GPT-4o mini model in order to adapt it to a set of document(s) and specific domain. **This step is critical as the quality of your training data will greatly influence the performance of your fine-tuned model.**

For RAFT, these are the different steps to prepare the training dataset:
- Collect Domain-Specific Documents: Gather documents relevant to the domain you want to specialize the LLM in (e.g., medical documents for PubMed, legal documents, API documentation for software).
- Chunk the file into Documents
- For each Document chunk, generate a set of Questions that can be answered from the Document
- For each Document-Question pair, create a list of documents using:
    - **Golden Document (D*)**: Document that contains the answer to the question.
    - **Distractor Documents (Dk)**: Documents that do not contain relevant information.
- Question-Answer-Document Triplets: From each **Document-Question** pair, generate a factual **Answer** based on the Golden Document.

Curating a good training dataset often involves manual work and review by SMEs. That said, we can use an LLM to help us generate an initial set of training examples that can be vetted and further refined by SMEs.


### 0. Pre-requisites

For this hands-on workshop, all you need is access to an Azure subscription and the ability to create Azure OpenAI resources and deployments. 

0. Install poppler for PDF processing

- on Linux run `sudo apt-get install -y poppler-utils`
- on Mac run `brew install poppler`
- on Windows run `conda install -c conda-forge poppler`

1. Create a code environment and install the necessary packages

```shell
conda create -n raft python=3.11

conda activate raft

pip install -r requirements.txt
```

2. Create a GPT-4o deployment
3. Create a GPT-4o mini deployment
4. Create an Azure OpenAI resource in North Central US or Sweden Central (regions where gpt-4o-mini fine tuning is supported)
5. Create a `.env` file based on the [sample.env](./sample.env) file in this repository to store your credentials and important environment variables. Paste your AOAI endpoints, keys and deployment names, name the file `.env`

**Import libraries**

In [1]:
from openai import AzureOpenAI
from dotenv import load_dotenv
from io import BytesIO
import base64
from typing import Literal, Any
import os
from math import ceil
import random
from tqdm import tqdm

load_dotenv()

generator_client = AzureOpenAI(
    azure_endpoint=os.getenv("AOAI_GPT4o_ENDPOINT"),
    api_version="2024-02-01",
    api_key=os.getenv("AOAI_GPT4o_API_KEY")
)

gpt4o_deployment = os.getenv("AOAI_GPT4o_DEPLOYMENT")


### 1. Loading and chunking domain-specific documents

For Retrieval Augmented Fine Tuning, we need to generate Question-Documents-Answer triplets. The first step is to create document chunks based on our domain-specific documents we want to specialize our model on.

In most cases, the documents have already been indexed in a vector database like Azure AI Search.

In this example, we will retrieve all documents from Azure AI Search and generate training data based on the content of the index.

In [3]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import random

class AzureSearchClient:
    def __init__(self, endpoint, index_name, api_key):
        """
        Initializes the AzureSearchClient with the given endpoint, index name, and API key.

        :param endpoint: The endpoint of the Azure Search service.
        :param index_name: The name of the index to search.
        :param api_key: The API key for the Azure Search service.
        """
        self.endpoint = endpoint
        self.index_name = index_name
        self.api_key = api_key
        self.client = SearchClient(endpoint=self.endpoint,
                                   index_name=self.index_name,
                                   credential=AzureKeyCredential(self.api_key))

    def get_all_documents(self):
        """
        Retrieves all documents from the Azure Search index using pagination.

        :return: A list of all documents in the index.
        """
        all_documents = []
        page_size = 1000   # Maximum page size
        skip = 0
        total_count = None

        # Initial search to get total count
        results = self.client.search(
            search_text="*",
            top=0,
            include_total_count=True
        )
        total_count = results.get_count()
        if total_count is None:
            raise Exception("Unable to retrieve the total count of documents.")
        print(f"Total documents found: {total_count}")

        while skip < total_count:
            results = self.client.search(
                search_text="*",
                top=page_size,
                skip=skip
            )
            documents = list(results)
            all_documents.extend(documents)
            skip += page_size
            print(f"Fetched {len(all_documents)} of {total_count} documents.")

        return all_documents

    def get_random_sample(self, n):
        """
        Returns a random sample of n documents from the index.

        :param n: The number of documents to sample.
        :return: A list of n randomly sampled documents.
        """
        all_documents = self.get_all_documents()
        if n > len(all_documents):
            raise ValueError(f"Requested sample size {n} exceeds the total number of documents {len(all_documents)}.")
        return random.sample(all_documents, n)



In [5]:
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_KEY")
search_index = "tax-fr"

client = AzureSearchClient(search_endpoint, search_index, search_key)

docs = client.get_all_documents()
print(f'fetched {len(docs)} documents')

Total documents found: 1572
Fetched 1000 of 1572 documents.
Fetched 1572 of 1572 documents.
fetched 1572 documents


{'url': 'https://vinceprojectdata.blob.core.windows.net/fileupload-tax-fr/impots_fr.pdf',
 'filepath': 'impots_fr.pdf',
 'contentVector': [-0.017528005,
  -0.01759297,
  0.016787384,
  -0.052363127,
  -0.031132022,
  0.010329699,
  -0.006587619,
  -0.016306631,
  -0.008322229,
  -0.013772932,
  0.032925103,
  0.03614745,
  -0.0046516126,
  0.005655348,
  -0.005356501,
  0.01774889,
  0.02842941,
  -0.027078103,
  0.02305017,
  -0.012674995,
  -0.0037355828,
  -0.009147305,
  -0.04485298,
  0.0019018989,
  -0.0112457285,
  -0.00035163204,
  0.013824905,
  -0.0061296043,
  0.015890844,
  0.0023534172,
  0.022400504,
  0.011486106,
  -0.009991872,
  -0.006626599,
  -0.032041557,
  -0.0025288272,
  0.016098738,
  0.018229645,
  0.030742222,
  -0.018476518,
  0.025752783,
  0.008835466,
  -0.012928365,
  -0.0024752298,
  0.014435591,
  0.018879311,
  0.0015137232,
  -0.017125212,
  -0.020451505,
  0.021594917,
  0.003258078,
  0.02102321,
  -0.029702757,
  0.011674508,
  -0.01654051,
  -0.0

In [7]:
content_field_name = 'content'
chunks = [doc.get(content_field_name) for doc in docs]

### 2. Generate training data from the retrieved documents

We define 2 main functions to generate our Question-Document-Answer triplets from our chunked document

1. `generate_instructions_gen()`: This function generates a list of questions based on an input document chunk
2. `generate_label()`: This function generates an Answer based on a Question-Document chunk pair

**a. First, lets look at the `generate_instructions_gen()` function on a sample**

In [8]:
def strip_str(s: str) -> str:
    """
    Helper function for helping format strings returned by GPT-4o.
    
    Parameters:  
    s (str): The input string to be formatted.  
  
    Returns:  
    str: A formatted string 
    """
    l, r = 0, len(s)-1
    beg_found = False
    for i in range(len(s)):
        if s[i].isalpha():
            if not beg_found:
                l = i
                beg_found = True
            else:
                r = i 
    r += 2
    return s[l:min(r, len(s))]

def generate_instructions_gen(client: AzureOpenAI, chunk: Any, x: int = 5, model: str = None) -> list[str]:
    """
    Generates a list of questions or use cases based on a provided chunk of context using an Azure OpenAI model.  

    Parameters:  
    client (AzureOpenAI): An instance of the Azure OpenAI client used to communicate with the OpenAI API.  
    chunk (Any): The context or chunk of text based on which the questions are to be generated.  
    x (int, optional): The number of questions to generate. Default is 5.  
    model (str, optional): The specific model to use for generating the questions. Default is None, which uses the default model configured in the client.  
  
    Returns:  
    list[str]: A list of generated questions.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a synthetic question-answer pair generator. Given a chunk of context about some topic(s), generate exactly %s example questions a user could ask and would be answered using information from the chunk. For example, if the given context was a Wikipedia paragraph about the United States, an example question could be 'How many states are in the United States?'" % (x)},
            {"role": "system", "content": "The questions should be able to be answered in a few words or less. Include only the questions in your response."},
            {"role": "user", "content": str(chunk)}
        ]
    )

    queries = response.choices[0].message.content.split('\n')
    queries = [strip_str(q) for q in queries]
    queries = [q for q in queries if any(c.isalpha() for c in q)]
    return queries[:int(x)]

Let's visualize an example picked randomly from our Document chunks

In [9]:
sample_index = random.randint(0, len(chunks)-1)
chunk = chunks[sample_index]

queries = generate_instructions_gen(generator_client, chunk, x=5, model=gpt4o_deployment)

In [10]:
print(chunk)

- Dernière modification le 01 janvier 2024 - Document généré le 04 janvier 2024

Le caractère libératoire du prélèvement ne peut être invoqué pour les produits qui sont pris en compte pour
la détermination du bénéfice imposable d'une entreprise industrielle, commerciale, artisanale ou agricole ou
d'une profession non commerciale.
 

Le taux du prélèvement est fixé :
 

a. A 45 % lorsque la durée du contrat a été inférieure à deux ans ; ce taux est de 35 p. 100 pour les contrats
souscrits à compter du 1er janvier 1990 ;
 

b. A 25 % lorsque cette durée a été égale ou supérieure à deux ans et inférieure à quatre ans ; ce taux est de
35 p. 100 pour les contrats souscrits à compter du 1er janvier 1990.
 

c. A 15 % lorsque cette durée a été égale ou supérieure à quatre ans.
 

d. A 7,5 % lorsque cette durée a été égale ou supérieure à six ans pour les bons ou contrats souscrits entre
le 1er janvier 1983 et le 31 décembre 1989 et à huit ans pour les contrats souscrits à compter du 1er janvi

In [11]:
queries


['Quel est le taux de prélèvement pour un contrat de durée inférieure à deux ans souscrit avant le 1er janvier ',
 "À partir de quelle date la disposition relative à la durée moyenne pondérée n'est-elle pas applicable aux contrats ",
 'Quel est le taux de prélèvement pour un contrat souscrit à compter du 1er janvier 1990 si sa durée est égale ou supérieure à huit ans ',
 'Selon le document, à quel taux le prélèvement est-il fixé pour les produits de contrats de capitalisation souscrits après le 27 septembre ',
 "Dans quel cas l'excédent du prélèvement sur l'impôt sur le revenu est-il restitué "]

**b. Generating questions, answers and adding distractor documents** 

In [ ]:
from datasets import Dataset, load_dataset
import random
from typing import Any

def encode_question_gen(question: str, chunk: Any) -> list[str]:
    """
    Encode multiple prompt instructions into a single string for the general case (`pdf`, `json`, or `txt`).

    Parameters:  
    question (str): The question to be answered.  
    chunk (Any): The context or chunk of text that provides the information needed to answer the question.  
  
    Returns:  
    list[str]: A list of messages formatted for the language model API, including system and user roles.  
    """
    
    prompts = []
        
    prompt = """
        Question: {question}\n Context: {context}\n
        Answer this question using the information given in the context above and no prior knowledge. Here is things to pay attention to: 
        - First provide step-by-step reasoning on how to answer the question. 
        - In the reasoning, if you need to copy paste some sentences from the context, include them in ##begin_quote## and ##end_quote##. This would mean that things outside of ##begin_quote## and ##end_quote## are not directly copy paste from the context. 
        - End your response with final answer in the form <ANSWER>: $answer, the answer should be given in a joyful and friendly tone.
        - If the answer cannot be found in the context, say "I'm sorry, I cannot answer this question as I'm missing the required information"
        You MUST begin your final answer with the tag "<ANSWER>:".
    """.format(question=question, context=str(chunk))
    prompts.append({"role": "system", "content": "You are a helpful question answerer who can provide an answer given a question and relevant context."})
    prompts.append({"role": "user", "content": prompt})
    return prompts

def generate_label(client: AzureOpenAI, question: str, context: Any, model: str = None) -> str | None:
    """
    Generates the label / answer to `question` using `context` and GPT-4o.

    Parameters:  
    client (AzureOpenAI): An instance of the Azure OpenAI client used to communicate with the OpenAI API.  
    question (str): The question to be answered.  
    context (Any): The context or chunk of text that provides the information needed to answer the question.  
    model (str, optional): The specific model to use for generating the answer. Default is None, which uses the default model configured in the client.  
  
    Returns:  
    str | None: The generated answer from the language model, or None if no answer was generated.
    """
    question = encode_question_gen(question, context)
    response = client.chat.completions.create(
        model=model,
        messages=question,
        n=1,
        temperature=0
    )
    response = response.choices[0].message.content
    return response

def add_chunk_to_dataset(
    client: AzureOpenAI,
    chunks: list[str], 
    chunk: str, 
    x: int = 5, 
    num_distract: int = 3, 
    p: float = 0.8,
    model: str = None
) -> None:
    """
    Given a chunk, create {Q, A, D} triplets and add them to the dataset.

     Parameters:  
    client (AzureOpenAI): An instance of the Azure OpenAI client used to communicate with the OpenAI API.  
    chunks (list[str]): A list of chunks of text from which distractor documents can be sampled.  
    chunk (str): The chunk of text to use as the primary context for generating questions and answers.  
    x (int, optional): The number of questions to generate for the given chunk. Default is 5.  
    num_distract (int, optional): The number of distractor documents to include with each question. Default is 3.  
    p (float, optional): The probability of including the oracle (original) document as part of the context. Default is 0.8.  
    model (str, optional): The specific model to use for generating questions and answers. Default is None, which uses the default model configured in the client. 
    """
    global ds
    global errors
    i = chunks.index(chunk)
    try:
        qs = generate_instructions_gen(client, chunk, x, model)
    except Exception as e:
        errors.append(e)
        return None
    for q in qs:
        datapt = {
            "id": None,
            "type": None,
            "question": None,
            "context": None,
            "oracle_context": None,
            "cot_answer": None
        }

        datapt["id"] = f"seed_task_{i}"
        datapt["type"] = "general"
        datapt["question"] = q

        # add num_distract distractor docs
        docs = [chunk]
        indices = list(range(0, len(chunks)))
        indices.remove(i)
        for j in random.sample(indices, num_distract):
            docs.append(chunks[j])
        
        # decides whether to add oracle document
        oracle = random.uniform(0, 1) < p
        if not oracle:
            docs[0] = chunks[random.sample(indices, 1)[0]]
        random.shuffle(docs)

        d = {
            "title": [],
            "sentences": []
        }

        d["title"].append(["placeholder_title"]*(num_distract+1))
        d["sentences"].append(docs)
        datapt["context"] = d
        datapt["oracle_context"] = chunk

        # add answer to q
        try:
            datapt["cot_answer"] = generate_label(client, q, chunk, model=model)
        except Exception as e:
            errors.append(e)
            continue

        # construct model instruction 
        context = ""
        for doc in docs:
            context += "<DOCUMENT>" + str(doc) + "</DOCUMENT>\n"
        context += q
        datapt["instruction"] = context

        # add to dataset
        if not ds:
            # init ds
            datapt["id"] = [datapt["id"]]
            datapt["type"] = [datapt["type"]]
            datapt["question"] = [datapt["question"]]
            datapt["context"] = [datapt["context"]]
            datapt["oracle_context"] = [datapt["oracle_context"]]
            datapt["cot_answer"] = [datapt["cot_answer"]]
            datapt["instruction"] = [datapt["instruction"]]
            ds = Dataset.from_dict(datapt)
        else:
            ds = ds.add_item(datapt)

**Let's execute this function a in multi-threaded way to speed up the process**

In [ ]:
from tqdm import tqdm
import concurrent.futures

errors = []
ds = Dataset.from_dict({})


def process_chunk(chunk):
    add_chunk_to_dataset(generator_client, chunks, chunk, 5, 3, model=gpt4o_deployment)

# Create a ThreadPoolExecutor with the desired number of workers
with concurrent.futures.ThreadPoolExecutor() as executor:
    # Submit the tasks to the executor and store the Future objects
    futures = [executor.submit(process_chunk, chunk) for chunk in chunks]

    # Use tqdm to create a progress bar
    with tqdm(total=len(chunks), desc="Processing chunks") as pbar:
        # Iterate over the completed futures as they become available
        for future in concurrent.futures.as_completed(futures):
            # Get the result of the completed future
            result = future.result()
            # Update the progress bar
            pbar.update(1)

# Print any errors that occurred during processing
print(f'Number of processing errors: {errors}/{len(chunks)}')

In [ ]:
training_df = ds.to_pandas()

print(f'{training_df.shape[0]} rows and {training_df.shape[1]} columns in the training dataset')

In [ ]:
# Previewing the generated data

training_df.head(2)

**c. Formatting the data in chat format for fine tuning with Azure OpenAI**

The conversational chat format is required to fine-tune gpt-4o-mini

In [ ]:
training_df["messages"] = training_df.apply(lambda x: [
                                                     {"role":"user", "content":x['instruction']},
                                                     {"role":"assistant", "content":x['cot_answer']}
                                                     ], axis=1)

In [ ]:
training_df.messages.values[12]

In [ ]:
training_df.dropna(subset=['cot_answer'], inplace=True)

### 2. Spitting our data into training and test sets

Splitting your data into training, validation and testing sets when fine-tuning a large language model (LLM) is crucial for ensuring the model's performance and generalization capabilities. The training set is used to teach the model, allowing it to learn patterns from the data. The validation set is used to track performance metrics during the training to avoid underfitting / overfitting. However, to objectively evaluate how well the model has learned and to ensure it can generalize to unseen data, a separate testing set is necessary. 

We will use this test set in order to measure the improvement of performance we get from using RAFT over RAG with gpt-4o-mini

In [ ]:
import numpy as np 

train_df, validate_df, test_df = np.split(
    training_df.sample(frac=1, random_state=42), 
                       [int(.8*len(training_df)), int(.9*len(training_df))]
                       )

print(f"Train: {train_df.shape[0]}, Validate: {validate_df.shape[0]}, Test: {test_df.shape[0]}")

In [ ]:

if not os.path.exists("./data/training_data"):
    os.makedirs("./data/training_data")

train_df[['messages']].to_json("./data/training_data/banking_train.jsonl", orient="records", lines=True)
test_df.to_json("./data/training_data/banking_test.jsonl", orient="records", lines=True)
validate_df[['messages']].to_json("./data/training_data/banking_validation.jsonl", orient="records", lines=True)

#### Congrats! We now have a labelled training dataset and a test dataset to evaluate our model's performance. Now go to the [finetuning notebook](./raft_finetuning.ipynb)